I made this code for assignment 2, with the aim of reproducing a part of the experiment from Li et al. (2020), COPOD: Copula-Based Outlier Detection. The experiment tests COPOD on three datasets used in the original paper: BreastW, Ionosphere, and Wine.
Original code and data: https://github.com/winstonll/COPOD

In [2]:
#Installing the library and downloading the data

%pip -q install pyod==2.0.7

# downloading the original datasets from the authors' GitHub
!test -d COPOD || git clone -q https://github.com/winstonll/COPOD.git

#importing
import numpy as np
import pandas as pd
from scipy.io import loadmat
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
from pyod.models.copod import COPOD

results = []

# testing COPOD on 3 datasets from the paper
for dataset in ["breastw", "ionosphere", "wine"]:

    # loading data set
    data = loadmat("COPOD/data/" + dataset + ".mat")
    X = data["X"]           # Features
    y = data["y"].ravel()   # Labels (0 = normal, 1 = outlier)

    contamination = np.mean(y)  # Percentage of outliers

    # repeating the experiment 10 times, like in the paper
    for seed in range(10):

        # using 60% for training and 40% for testing
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.4, random_state=seed
        )

        # standardizing the data
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        # training the model
        model = COPOD(contamination=contamination)
        model.fit(X_train)

        # calculate outlier scores
        scores = model.decision_function(X_test)

        # calculaye roc and ap
        roc = roc_auc_score(y_test, scores)
        ap = average_precision_score(y_test, scores)

        # save muy results
        results.append({
            "Dataset": dataset,
            "ROC-AUC": roc,
            "AP": ap
        })

# calculate average results over the 10 runs
table = pd.DataFrame(results).groupby(
    "Dataset", as_index=False
).mean()

# COPOD results from paper
paper_roc = {"breastw": 0.9936, "ionosphere": 0.8307, "wine": 0.9490}
paper_ap = {"breastw": 0.9877, "ionosphere": 0.7194, "wine": 0.6082}

# Compare my results with those from the paper
table["Paper ROC-AUC"] = table["Dataset"].map(paper_roc)
table["Paper AP"] = table["Dataset"].map(paper_ap)

# print the results in a table so it's easy to compare
print("Comparison with the paper's results:")
print(table.round(4).to_string(index=False))






Comparison with the paper's results:
   Dataset  ROC-AUC     AP  Paper ROC-AUC  Paper AP
   breastw   0.9935 0.9877         0.9936    0.9877
ionosphere   0.7992 0.6808         0.8307    0.7194
      wine   0.8909 0.4324         0.9490    0.6082
